## loading the libraries and dataset

In [ ]:
import pandas as pd
import json

# Load the train.json file
data_path = "train.json"
data = []

with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            # skip or inspect broken line
            continue

df = pd.DataFrame(data)
# print(train_df.head())
df

records = []
for col in df.columns:
    # Extract the dict from each cell in that column
    records.extend(df[col].dropna().tolist())

# Step 2: Convert list of dicts into a DataFrame
train_df = pd.DataFrame(records)

# Step 3: Preview
train_df
# print(flat_df.head())
# print(flat_df.shape)


,answer,type,context,question
0,"In 1512, Parliament passed a significant act t...",factual,During the Hundred Years' War a French attack ...,In what year did Parliament pass a notable law...
1,The Spanish and French were the ones who estab...,factual,"""By May 1539, Conquistador Hernando de Soto sk...",Who established early settlements in Florida
2,"Traditionally, monsoons in Punjab are expected...",factual,The onset of the southwest monsoon is anticipa...,When do monsoons traditionally happen in Punjab?
3,The media made the requests for Kondo to use o...,factual,Media requests at the trade show prompted Kond...,Who made the requests for Kondo to use orchest...
4,According to historians Robert Friedel and Pau...,factual,In addressing the question of who invented the...,How many inventors came up with electric lamps...
...,...,...,...,...
21016,"At the first proceeding, the jury determines w...",factual,"In 1976, contemporaneously with Woodson and Ro...",What is decided at the first proceeding?
21017,King Henry VII of England commissioned John Ca...,factual,The foundations of the British Empire were lai...,Who commissioned John Cabot's voyage?
21018,The distinguishing visual design feature of Fr...,irrelevant,The fossil record suggests that the last few m...,What suggests that the last few million years ...
21019,"For group law and topology to integrate well, ...",irrelevant,"Following the highly publicized incident, West...",What industry did Kanye turn to after taking a...


In [ ]:
import pandas as pd
import json

# Load the test.json file
data_path = "test.json"

with open(data_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
test_df = pd.DataFrame(data)

test_df

,ID,answer,type,context,question
0,1,"In the mid-19th century, the Bronx was referre...",,The Bronx street grid is irregular. Like the n...,What was the Bronx called in the mid-19th cent...
1,2,"Beyoncé's father, Mathew Knowles, began managi...",,,When did Beyoncé begin to manage the girl group?
2,3,Dionysus was similar to the Roman god Bacchus.,,"While the new plebeian nobility made social, p...",To what Roman god was Dionysus similar?
3,4,The most crucial defense in preventing the spr...,,"Techniques like hand washing, wearing gowns, a...",What is the most important defense against the...
4,5,The third largest long term acute care provide...,,The Baylor College of Medicine has annually be...,What is the third largest acute care center in...
...,...,...,...,...,...
1995,1996,Modern dogs likely originated when human being...,,,Modern dogs likely began when human beings wer...
1996,1997,The bird that is mentioned in the Book of Job ...,,Records of bird migration were made as much as...,What bird is mentioned in the book of Job?
1997,1998,"Pope Pius XII passed away on October 9, 1958.",,Following the death of Pope Pius XII on 9 Octo...,When did Pope Pius XII die?
1998,1999,The verse in the Quran that expresses the nece...,,Shias believe that Imamah is of the Principles...,What verse in the quran expresses the necessit...


## EDA, metric harness, baselines

- EDA - class imbalance, empty / missing context handling, length distributions, lexical overlap patterns among Q, C, A.

- Macro precision/recall/F1 per class / custom weighted confusion matrix and overall weighted score.

- Heuristic + similarity baseline
  - Features - TF-IDF cosine/ BM25 between (A, Q), (A, C), and (Q,C) ; length deltas; negation words; contradiction cue words.

- Off the shelf NLI (zero shot)
  - Treat premise = context, hypothesis = answer -> map MBLI labels: entailment -> Factual, contradiction -> condradiction, neutral -> irrelevant
  - try Deberta-v3-large-MNLI, roberta-large-MBLI.

### EDA

In [ ]:
train_df[train_df.apply(lambda x: x.context.__len__() == 0, axis = 1)].type.value_counts()/ (1593+150+146)

In [ ]:
train_df[train_df.apply(lambda x: x.context.__len__() == 0, axis = 1)].type.value_counts()/ (1593+150+146)

,count
type,
factual,0.832716
irrelevant,0.079407
contradiction,0.077290


In [ ]:
print('question: ', test_df.loc[1].question, '\nanswer: ', test_df.loc[1].answer)

question:  When did Beyoncé begin to manage the girl group? 
answer:  Beyoncé's father, Mathew Knowles, began managing the girl group in 1995 when he resigned from his job to focus on their career.


In [ ]:
# EDA - class imbalance, empty / missing context handling, length distributions

# Class imbalance
print("Class distribution:")
display(train_df['type'].value_counts())

# Missing values
print("\nMissing values per column:")
display(train_df.isnull().sum())

# Length distributions of text columns
print("\nLength distribution of 'answer':")
display(train_df['answer'].str.len().describe())

print("\nLength distribution of 'context':")
display(train_df['context'].str.len().describe())

print("\nLength distribution of 'question':")
display(train_df['question'].str.len().describe())

Class distribution:


,count
type,
factual,17431
contradiction,1818
irrelevant,1772



Missing values per column:


,0
answer,0
type,0
context,0
question,0



Length distribution of 'answer':


,answer
count,21021.000000
mean,89.662718
std,44.490616
min,1.000000
25%,62.000000
50%,83.000000
75%,110.000000
max,642.000000



Length distribution of 'context':


,context
count,21021.000000
mean,686.349127
std,360.602114
min,0.000000
25%,526.000000
50%,665.000000
75%,867.000000
max,3706.000000



Length distribution of 'question':


,question
count,21021.000000
mean,59.381143
std,21.349175
min,12.000000
25%,44.000000
50%,56.000000
75%,71.000000
max,201.000000
